In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [2]:
df=pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df=df.drop(columns='customerID')

In [5]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [7]:
#df['TotalCharges'].fillna(df['TotalCharges'].median)

In [8]:
df['SeniorCitizen']=df['SeniorCitizen'].map({1:"Yes",0:"No"})
df['Churn']=df['Churn'].map({"Yes":1,"No":0})

In [9]:
X=df.drop(columns='Churn')
y=df['Churn']
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
df=df.drop(columns='Churn')

In [11]:
nums=[x for x in df.select_dtypes(include=['int','float'])]
cats=[x for x in df.select_dtypes(include=['object'])]

In [12]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy='median')
X_train[nums]=imputer.fit_transform(X_train[nums])
X_test[nums]=imputer.transform(X_test[nums])

In [13]:
scaler=StandardScaler()
X_train[nums]=scaler.fit_transform(X_train[nums])
X_test[nums]=scaler.transform(X_test[nums])

In [14]:
encode=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
X_train_encode=encode.fit_transform(X_train[cats])
X_test_encode=encode.transform(X_test[cats])

In [15]:
X_train_encode=pd.DataFrame(
    X_train_encode,
    columns=encode.get_feature_names_out(cats),
    index=X_train.index
)
X_test_encode=pd.DataFrame(
    X_test_encode,
    columns=encode.get_feature_names_out(cats),
    index=X_test.index
)
    

In [16]:
X_train_final=pd.concat([X_train[nums],X_train_encode],axis=1)
X_test_final=pd.concat([X_test[nums],X_test_encode],axis=1)

In [17]:
model=LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

model.fit(X_train_final,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [18]:
predictions=model.predict(X_test_final)

In [19]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,predictions)
print(f'Accuracy : {accuracy}')

Accuracy : 0.7374024130589071


In [20]:
from sklearn.ensemble import RandomForestClassifier

RF=RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)

RF.fit(X_train_final,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
RF_preds=model.predict(X_test_final)

In [22]:
acc=accuracy_score(y_test,RF_preds)
print(f'Accuracy : {acc}')

Accuracy : 0.7374024130589071


In [23]:
from sklearn.metrics import confusion_matrix
confusion=confusion_matrix(y_test,predictions)
print(confusion)

[[746 289]
 [ 81 293]]


In [24]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409



In [27]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5]
}


grid = GridSearchCV(
    estimator=RF,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

In [28]:
grid.fit(X_train_final, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [3, 5, ...], 'min_samples_split': [2, 5]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [37]:
#Gridsearch
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_final, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,LogisticRegre...max_iter=1000)
,param_grid,"{'C': [0.01, 0.1, ...], 'solver': ['liblinear', 'lbfgs']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [38]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test_final)

Best Parameters: {'C': 0.1, 'solver': 'lbfgs'}
Best CV Score: 0.7656029954462985


In [34]:
#Random search
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"],
    "tol": [0.0001, 0.001, 0.01]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_final, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

best_model = random_search.best_estimator_

{'tol': 0.01, 'solver': 'lbfgs', 'C': 100}
0.7698758299898876
